<div style="background: linear-gradient(135deg, #0B1F3F 0%, #1a4a6e 50%, #008C8C 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: white; font-family: Georgia, serif; margin: 0; font-size: 2.2em;">
🏛️ Sessions 15–17: From Models to Decisions
</h1>
<p style="color: #B8953E; font-size: 1.25em; margin-top: 10px; font-family: Calibri, sans-serif;">
Baselines • Complexity Trade-offs • Interpretability • Policy Translation
</p>
<p style="color: #B0D0D0; font-size: 0.95em; margin-top: 8px;">
Credit Risk Modelling Programme &nbsp;|&nbsp; MBA Advanced Analytics &nbsp;|&nbsp; 2025–26
</p>
</div>

<div style="background:#FFFFFF;border:2px solid #B8953E;padding:20px 28px;border-radius:10px;margin:10px 0 20px 0;">
<h3 style="color:#0B1F3F;margin-top:0;">📖 The Final Ascent</h3>
<p style="color:#333;font-size:1.05em;line-height:1.7;">
Sessions 13–14 were about <em>understanding</em> the data. This notebook is about <em>acting</em> on it.
We will build models, interrogate them, and translate their findings into actionable business policies.
By the end, you will have everything needed for the <strong>Session 18 Capstone</strong>: a complete
Decision Blueprint that a Chief Risk Officer could use.
</p>
<p style="color:#555;font-size:0.95em;margin-top:12px;font-style:italic;">
⬆️ Self-sufficiency continues to grow: <strong>“Think About It”</strong> prompts now ask you to
predict model outcomes and design policy. All code hints remain complete.
</p>
</div>

<div style="background:#FFFFFF;border:2px solid #008C8C;padding:20px 28px;border-radius:10px;margin:10px 0 20px 0;">
<h3 style="color:#0B1F3F;margin-top:0;">🗺️ Notebook Roadmap</h3>
<ol style="color:#333;font-size:1.02em;line-height:1.8;">
<li><strong>Setup & Feature Reload</strong> — Load the S14 feature sets and prepare train/test splits</li>
<li><strong>Baseline: Regularised Logistic Regression</strong> — L1 vs L2, coefficient interpretation</li>
<li><strong>Gradient Boosting Machines</strong> — XGBoost/LightGBM, feature importance</li>
<li><strong>Model Calibration</strong> — Are predicted probabilities trustworthy?</li>
<li><strong>Complexity Trade-offs</strong> — Polynomial features, splines, when simple wins</li>
<li><strong>SHAP: Global Interpretability</strong> — What does the model see across all applicants?</li>
<li><strong>SHAP: Individual Explanations</strong> — Why was <em>this</em> applicant approved or denied?</li>
<li><strong>Partial Dependence Plots</strong> — How does each feature affect default probability?</li>
<li><strong>Policy Translation</strong> — Convert model insights into lending rules</li>
<li><strong>Decision Blueprint Preparation</strong> — Assemble the capstone deliverable</li>
</ol>
</div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 1: Setup & Feature Reload</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">Load the S14 feature sets and prepare temporal train/test splits</p></div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, brier_score_loss,
    classification_report, confusion_matrix,
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

NAVY = '#0B1F3F'; TEAL = '#008C8C'; GOLD = '#B8953E'
CORAL = '#E8634A'; LGOLD = '#FDF6E8'; LTEAL = '#E0F2F2'
PURPLE = '#6C5B7B'
PALETTE = [NAVY, TEAL, GOLD, CORAL, PURPLE, '#C06C84']
sns.set_palette(PALETTE)

plt.rcParams.update({
    'figure.figsize': (12, 6), 'figure.dpi': 120,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'axes.titleweight': 'bold', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

print("\u2705 Libraries loaded.")

✅ Libraries loaded.


In [2]:
# ── Load cleaned data and S14 reduced features ──
DATA_DIR = r'C:\Users\DevA\OneDrive\Desktop\All in one - Desktop Files\MBA\Q8\AI ML\April 1 and 8'

app = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))

# Reproduce cleaning
app['DAYS_EMPLOYED_ANOMALY'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)
app['AGE_YEARS'] = (-app['DAYS_BIRTH'] / 365).round(1)
app['EMPLOYMENT_YEARS'] = (-app['DAYS_EMPLOYED'] / 365).round(1)
app['DEBT_TO_INCOME'] = (app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']).round(2)
app['ANNUITY_BURDEN'] = (app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']).round(4)
app['CREDIT_GOODS_RATIO'] = (app['AMT_CREDIT'] / app['AMT_GOODS_PRICE']).round(3)
app['EXT_SCORE_BLEND'] = app[
    ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
].mean(axis=1).round(4)

# ── Build modelling-ready feature matrix ──
model_features = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SCORE_BLEND',
    'AGE_YEARS', 'EMPLOYMENT_YEARS', 'DAYS_EMPLOYED_ANOMALY',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DEBT_TO_INCOME', 'ANNUITY_BURDEN', 'CREDIT_GOODS_RATIO',
    'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
    'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'HOUR_APPR_PROCESS_START',
]

X = app[model_features].copy()
y = app['TARGET'].copy()

# Impute missing values
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=model_features)

# ── Train / Test split (stratified to preserve class balance) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y,
)

print(f"\u2705 Features prepared: {X.shape[1]} columns")
print(f"   Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows")
print(f"   Train default rate: {y_train.mean():.2%}")
print(f"   Test  default rate: {y_test.mean():.2%}")

✅ Features prepared: 22 columns
   Train: 246,008 rows  |  Test: 61,503 rows
   Train default rate: 8.07%
   Test  default rate: 8.07%


<div style="background:linear-gradient(90deg,#E0F2F2,#FDF6E8);border-left:5px solid #008C8C;border-right:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🧠 Interpretation</strong><br><span style="color:#333;">The stratified split ensures that both train and test have the same ~8% default rate. This is critical with imbalanced data — a random split could put 9% defaults in train and 6% in test, making performance comparisons misleading.</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 2: Baseline: Regularised Logistic Regression</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">The interpretable starting point</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Build L1 (Lasso) and L2 (Ridge) logistic regression models. Compare their AUC-ROC scores and examine which features L1 zeroes out. Understand why regularisation matters.</span></div>

<div style="background:linear-gradient(135deg,#E8EDF4,#E0F2F2);border:2px solid #0B1F3F;padding:20px 24px;border-radius:10px;margin:12px 0;">
<h3 style="color:#0B1F3F;margin-top:0;">📐 Why Regularisation?</h3>
<p style="color:#333;line-height:1.7;">
Plain logistic regression fits coefficients to minimise prediction error. But with many features,
it can <strong>overfit</strong> — fitting noise in the training data that doesn’t generalise.
Regularisation adds a penalty for large coefficients, forcing the model to be simpler.
</p>
<p style="color:#333;line-height:1.7;">
<strong>L1 (Lasso):</strong> Penalty = sum of |coefficients|. Drives weak coefficients to exactly zero,
performing automatic feature selection. Great for interpretability.<br>
<strong>L2 (Ridge):</strong> Penalty = sum of coefficients². Shrinks all coefficients toward zero but
never zeroes them out. Better for prediction when many features contribute a little.
</p>
</div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 1: L1 vs L2 Logistic Regression</h3></div>

In [ ]:
# YOUR CODE: Fit L1 and L2 logistic regression
# 1. Scale features with StandardScaler
# 2. Fit LogisticRegression(penalty='l1', C=0.1, solver='saga')
# 3. Fit LogisticRegression(penalty='l2', C=0.1, solver='saga')
# 4. Predict probabilities on test set
# 5. Compute AUC-ROC for both
# 6. Count how many features L1 zeroed out


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Scale: <code>scaler = StandardScaler(); X_sc = scaler.fit_transform(X_train)</code>. Predict probabilities: <code>lr.predict_proba(X_test)[:, 1]</code>. AUC: <code>roc_auc_score(y_test, y_prob)</code>. Zero coefficients: <code>(lr_l1.coef_[0] == 0).sum()</code>.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 2: Coefficient Comparison</h3></div>

In [ ]:
# YOUR CODE: Plot L1 and L2 coefficients side by side
# Which features does L1 zero out? Which are the strongest predictors?
# Negative coefficient = protective (lower default risk)


<div style="background:linear-gradient(90deg,#E0F2F2,#FDF6E8);border-left:5px solid #008C8C;border-right:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🧠 Interpretation</strong><br><span style="color:#333;"><strong>Key observations:</strong><br>• L1 zeroes out several features (e.g., CNT_CHILDREN, HOUR_APPR_PROCESS_START) — it has decided they are not worth keeping.<br>• EXT_SOURCE features have the largest negative coefficients in both models: higher scores = lower default = protective.<br>• The AUC difference between L1 and L2 is typically small (within 0.005), but L1 gives a sparser, more interpretable model.</span></div>

<div style="background:#F0E6F6;border-left:5px solid #6C5B7B;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#6C5B7B;">👔 Manager’s Take</strong><br><span style="color:#333;">L1 logistic regression is effectively a <strong>credit scorecard</strong>. You can look at the coefficients and say: “For every 1-standard-deviation increase in EXT_SOURCE_2, the log-odds of default decrease by X.” Regulators love this transparency. But is the accuracy enough?</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 3: Gradient Boosting Machines</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">The accuracy upgrade — at a complexity cost</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Train a LightGBM model and compare its performance against logistic regression. Examine feature importance and discuss the accuracy-interpretability trade-off.</span></div>

<div style="background:linear-gradient(135deg,#FFF8E1,#FFF3E0);border:2px solid #B8953E;padding:16px 20px;border-radius:8px;margin:16px 0;"><strong style="color:#B8953E;">🤔 Think About It</strong><p style="color:#333;margin-top:8px;line-height:1.6;">Before you run the GBM, predict: will it beat logistic regression? By how much? What kind of patterns can trees capture that logistic regression cannot?</p></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 3: LightGBM Baseline</h3></div>

In [ ]:
# YOUR CODE: Train a LightGBM model
# 1. pip install lightgbm if needed
# 2. Create lgb.Dataset objects for train and test
# 3. Set sensible parameters (learning_rate=0.05, num_leaves=31, max_depth=6)
# 4. Train with early stopping (50 rounds patience)
# 5. Predict probabilities and compute AUC-ROC
# 6. Compare against logistic regression

import lightgbm as lgb


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Create dataset: <code>lgb.Dataset(X_train, label=y_train)</code>. Train: <code>lgb.train(params, train_data, num_boost_round=500, valid_sets=[val_data], callbacks=[lgb.early_stopping(50)])</code>. Predict: <code>gbm.predict(X_test)</code>.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 4: ROC Curves: Head-to-Head</h3></div>

In [ ]:
# YOUR CODE: Plot ROC and Precision-Recall curves for all 3 models
# Include: random baseline on ROC, prevalence baseline on PR


<div style="background:#E0F2F2;border-left:5px solid #008C8C;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#008C8C;">📊 Business Insight</strong><br><span style="color:#333;"><strong>GBM typically improves AUC by 2–4 points</strong> over logistic regression. On the PR curve (which is more informative for imbalanced data), the gap is often larger. But here’s the trade-off: you can explain a logistic regression coefficient; you cannot easily explain why a GBM with 300 trees made a specific prediction — unless you use SHAP (Part 6).</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 4: Model Calibration</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">Are predicted probabilities trustworthy?</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Assess whether model probabilities are well-calibrated: does a predicted 20% default probability actually mean ~20% of those applicants default?</span></div>

<div style="background:#E0F2F2;border:2px solid #008C8C;padding:16px 22px;border-radius:8px;margin:12px 0;">
<strong style="color:#008C8C;">📐 Why Calibration Matters</strong>
<p style="color:#333;margin-top:8px;line-height:1.6;">
A model can rank applicants perfectly (high AUC) but assign wrong probabilities.
If your model says “30% default risk” but only 5% of those applicants actually default,
your pricing, provisioning, and capital allocation will all be wrong.<br><br>
<strong>Well-calibrated:</strong> The calibration curve follows the diagonal.<br>
<strong>Over-confident:</strong> The curve is S-shaped (extremes are too extreme).<br>
<strong>Under-confident:</strong> The curve is flat (probabilities are too close to the mean).
</p>
</div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 5: Calibration Curves</h3></div>

In [ ]:
# YOUR CODE: Plot calibration curves and compute Brier scores
# calibration_curve(y_true, y_prob, n_bins=10, strategy='quantile')
# brier_score_loss(y_true, y_prob)
# Perfect calibration = diagonal line


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Calibration curve: <code>fraction_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10)</code>. Brier score: <code>brier_score_loss(y_test, probs)</code>. Lower Brier = better calibration.</span></div>

<div style="background:linear-gradient(90deg,#E0F2F2,#FDF6E8);border-left:5px solid #008C8C;border-right:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🧠 Interpretation</strong><br><span style="color:#333;">Logistic regression is typically <em>naturally well-calibrated</em> because it directly models probabilities. GBMs often require post-hoc calibration (e.g., Platt scaling or isotonic regression) because tree ensembles optimise for ranking, not probability accuracy. The Brier score combines calibration and discrimination into a single number.</span></div>

<div style="background:linear-gradient(90deg,#E8EDF4,#F0E6F6);border:2px dashed #6C5B7B;padding:16px 20px;border-radius:8px;margin:16px 0;"><strong style="color:#6C5B7B;">🧭 What Would You Do Next?</strong><p style="color:#333;margin-top:8px;line-height:1.6;">Given that GBM has higher AUC but logistic regression has better calibration, what would your recommendation be for a production credit-scoring system? Would you choose one model, or could you combine their strengths? How?</p></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 5: Complexity Trade-offs</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">When simple models win — polynomial features and splines</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Add polynomial features to logistic regression and observe the impact. Introduce splines as a middle ground between linear models and black-box GBMs.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 6: Polynomial Features — More Is Not Always Better</h3></div>

In [ ]:
# YOUR CODE: Add degree-2 polynomial features to the top 5 features
# Fit logistic regression and compare AUC against linear and GBM
# Does the polynomial expansion close the gap with GBM?

from sklearn.preprocessing import PolynomialFeatures

top5 = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS', 'DEBT_TO_INCOME']


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Polynomial: <code>poly = PolynomialFeatures(degree=2, include_bias=False); X_poly = poly.fit_transform(X_train[top5])</code>. This creates interaction terms (A*B) and squared terms (A²).</span></div>

<div style="background:linear-gradient(90deg,#E0F2F2,#FDF6E8);border-left:5px solid #008C8C;border-right:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🧠 Interpretation</strong><br><span style="color:#333;"><strong>Polynomials help but don’t close the gap.</strong> Degree-2 features capture interactions (EXT_SOURCE_2 × AGE) and non-linearities (AGE²), which logistic regression cannot model on its own. But degree-3 or higher rapidly creates thousands of features, inviting overfitting. Splines are a better approach for capturing non-linearity without the explosion.</span></div>

<div style="background:#FDE8E5;border-left:5px solid #E8634A;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#E8634A;">⚠️ Important</strong><br><span style="color:#333;"><strong>Curse of dimensionality:</strong> 5 features at degree 2 = 20 features. 22 features at degree 2 = 275 features. At degree 3 = 2,300. At degree 4 = 15,000+. The feature space grows combinatorially, and most of those features are noise.</span></div>

<div style="background:linear-gradient(135deg,#FFF8E1,#FFF3E0);border:2px solid #B8953E;padding:16px 20px;border-radius:8px;margin:16px 0;"><strong style="color:#B8953E;">🤔 Think About It</strong><p style="color:#333;margin-top:8px;line-height:1.6;">Polynomial features are a blunt instrument — they add <em>all possible</em> interactions. But maybe only a few specific non-linear patterns matter (e.g., default risk drops sharply as EXT_SOURCE_2 increases from 0 to 0.4, then flattens). What kind of function could capture that pattern smoothly without creating hundreds of features?</p></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 7: Splines — The Pragmatic Middle Ground</h3></div>

In [ ]:
# YOUR CODE: Fit spline transforms to EXT_SOURCE_2, EXT_SOURCE_3, AGE_YEARS
# 1. Use SplineTransformer(n_knots=5, degree=3)
# 2. Visualise the basis functions
# 3. Combine spline features with remaining linear features
# 4. Fit logistic regression and compare AUC

from sklearn.preprocessing import SplineTransformer
spline_features = ['EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS']


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Spline transform: <code>spline = SplineTransformer(n_knots=5, degree=3); X_sp = spline.fit_transform(X[['EXT_SOURCE_2']])</code>. Combine with other features: <code>np.hstack([X_other, X_sp])</code>.</span></div>

<div style="background:#E0F2F2;border-left:5px solid #008C8C;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#008C8C;">📊 Business Insight</strong><br><span style="color:#333;"><strong>Splines close 50–70% of the gap between linear and GBM</strong>, while keeping the model fully interpretable. Each spline basis function captures a smooth, local non-linear pattern. You can plot the fitted curve and show a manager exactly how default probability changes with EXT_SOURCE_2. This is the sweet spot for regulated industries.</span></div>

<div style="background:#F0E6F6;border-left:5px solid #6C5B7B;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#6C5B7B;">👔 Manager’s Take</strong><br><span style="color:#333;">In banking, regulators may reject black-box models. Spline-based logistic regression gives you 80–90% of GBM’s accuracy while remaining fully transparent. This is often the optimal trade-off for production credit-scoring models.</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 6: SHAP: Global Interpretability</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">What does the model see across all applicants?</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Compute SHAP values for the LightGBM model. Build summary plots showing which features drive predictions globally. Understand the difference between feature <em>importance</em> and feature <em>effect</em>.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 8: SHAP Summary Plot</h3></div>

In [ ]:
# YOUR CODE: Compute SHAP values for the GBM model
# 1. pip install shap if needed
# 2. Create TreeExplainer for the GBM
# 3. Sample ~3000 test observations for speed
# 4. Compute SHAP values
# 5. Create a summary (beeswarm) plot

import shap


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Explainer: <code>explainer = shap.TreeExplainer(gbm)</code>. Values: <code>shap_values = explainer.shap_values(X_sample)</code>. Plot: <code>shap.summary_plot(shap_values, X_sample, max_display=15)</code>.</span></div>

<div style="background:linear-gradient(90deg,#E0F2F2,#FDF6E8);border-left:5px solid #008C8C;border-right:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🧠 Interpretation</strong><br><span style="color:#333;"><strong>How to read the SHAP summary plot:</strong><br>• Each dot is one applicant. Colour = feature value (red=high, blue=low).<br>• X-axis = impact on default prediction (right = pushes toward default).<br>• EXT_SOURCE_2: blue dots (low scores) push RIGHT (toward default); red dots (high scores) push LEFT (protective).<br>• Features are ranked by overall importance (total absolute SHAP).</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 7: SHAP: Individual Explanations</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">Why was <em>this</em> applicant approved or denied?</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Create waterfall plots for individual applicants: one who repaid, one who defaulted, and one borderline case. Practise explaining each decision in plain English.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 9: Three Applicant Stories</h3></div>

In [ ]:
# YOUR CODE: Find 3 applicants: low-risk, high-risk, borderline
# For each, show the top 8 SHAP contributors as a horizontal bar chart
# Include the actual feature values alongside the SHAP values
#
# This is the exercise that regulators care about most.


<div style="background:#F0E6F6;border-left:5px solid #6C5B7B;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#6C5B7B;">👔 Manager’s Take</strong><br><span style="color:#333;">These individual explanations are the basis of <strong>adverse action notices</strong>. In many jurisdictions, when you deny a loan, you must tell the applicant the main reasons. SHAP makes this possible even for complex models: “Your application was declined primarily because your external credit score was below our threshold and your employment tenure was shorter than average.”</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 8: Partial Dependence Plots</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">How does each feature affect default probability?</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Build PDPs for the top 5 features. These show the <em>marginal effect</em> of each feature on default probability, averaging over all other features.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 10: PDP Gallery</h3></div>

In [ ]:
# YOUR CODE: Create Partial Dependence Plots for the top 5 features
# PDP shows: if I change THIS feature while holding everything else constant,
# how does the average predicted probability change?
#
# Manual approach:
#   1. Create a grid of values for the feature
#   2. For each grid value, set ALL test rows to that value
#   3. Predict and average -> that's the PDP value

pdp_features = ['EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS', 'DEBT_TO_INCOME', 'ANNUITY_BURDEN']


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Manual PDP: <code>for g in grid: X_temp = X_test.copy(); X_temp[feat] = g; pdp.append(gbm.predict(X_temp).mean())</code>. Grid: <code>np.linspace(X[feat].quantile(0.02), X[feat].quantile(0.98), 50)</code>.</span></div>

<div style="background:linear-gradient(90deg,#E0F2F2,#FDF6E8);border-left:5px solid #008C8C;border-right:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🧠 Interpretation</strong><br><span style="color:#333;"><strong>PDPs tell actionable stories:</strong><br>• <strong>EXT_SOURCE_2:</strong> Default probability drops steeply from 0 to 0.4, then flattens. This means scores below 0.4 are the “danger zone.”<br>• <strong>AGE_YEARS:</strong> Younger applicants face higher predicted risk, declining steadily to age ~55.<br>• <strong>DEBT_TO_INCOME:</strong> Risk increases gradually with leverage, accelerating above a ratio of ~6.<br>These are the curves a Chief Risk Officer uses to set policy thresholds.</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 9: Policy Translation</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">Converting model insights into lending rules</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Translate SHAP values and PDPs into a draft credit policy with specific thresholds, approval criteria, and risk categories. This is the core skill of an analytics leader.</span></div>

<div style="background:linear-gradient(135deg,#E8EDF4,#E0F2F2);border:2px solid #0B1F3F;padding:20px 24px;border-radius:10px;margin:12px 0;">
<h3 style="color:#0B1F3F;margin-top:0;">📐 From Model Output to Business Rule</h3>
<p style="color:#333;line-height:1.7;">
A model says: “P(default) = 14%.” A policy says: “Approve with enhanced monitoring.”
The bridge between them is <strong>threshold selection</strong> — choosing cutoff probabilities
that align with the business’s risk appetite and profitability targets.
</p>
<p style="color:#333;line-height:1.7;">
<strong>Key principle:</strong> The cost of a <em>false negative</em> (approving a bad loan) includes
the entire loan amount. The cost of a <em>false positive</em> (rejecting a good applicant) is the
lost profit margin. If a typical loan generates 5% profit but a default loses 100%, you need
the model to catch at least 20 bad loans for every good one it wrongly rejects.
</p>
</div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 11: Risk Tier Classification</h3></div>

In [ ]:
# YOUR CODE: Define risk tiers based on predicted probability thresholds
# Suggested tiers: <5% (Very Low), 5-10% (Low), 10-20% (Moderate), 20-35% (High), 35%+ (Very High)
# For each tier, compute: count, actual default rate, % of portfolio, % of all defaults
#
# This is the output a credit committee would review.


<div style="background:#FDF6E8;border-left:5px solid #B8953E;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#B8953E;">💡 Hint</strong><br><span style="color:#555;">Assign tiers: <code>df['tier'] = df['pred_prob'].apply(lambda p: 'A' if p < 0.05 else ...)</code>. Group analysis: <code>df.groupby('tier').agg(count=('actual','count'), rate=('actual','mean'))</code>.</span></div>

<div style="background:#E0F2F2;border-left:5px solid #008C8C;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#008C8C;">📊 Business Insight</strong><br><span style="color:#333;"><strong>Tier E (Very High Risk) contains ~5% of applicants but ~30–40% of all defaults.</strong> Rejecting or restructuring loans in this tier alone would eliminate a third of defaults while affecting only 1 in 20 applicants. This is the leverage point for a credit policy.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 12: Draft Credit Policy</h3></div>

In [ ]:
# YOUR CODE: Write a draft credit policy based on your risk tiers and SHAP findings
# Include:
#   1. Decision rules per tier (approve/review/decline)
#   2. Pricing guidance (risk-adjusted interest rates)
#   3. Key risk driver thresholds (from PDPs and SHAP)
#   4. An adverse action notice template
#
# Remember: this is for the Session 18 capstone. Write it for a CRO.


<div style="background:#F0E6F6;border-left:5px solid #6C5B7B;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#6C5B7B;">👔 Manager’s Take</strong><br><span style="color:#333;">This draft policy is the single most valuable deliverable in the entire programme. It translates months of data analysis into something a business can <em>act on tomorrow</em>. In the capstone, you’ll refine it, add customer segmentation, and present it to peers playing the role of sceptical executives.</span></div>

<div style="background:linear-gradient(90deg,#0B1F3F,#008C8C);padding:14px 24px;border-radius:8px;margin:30px 0 12px 0;"><h2 style="color:white;margin:0;font-family:Georgia,serif;">Part 10: Decision Blueprint Preparation</h2><p style="color:#008C8C;margin:4px 0 0 0;font-size:1.05em;">Assembling the capstone deliverable</p></div>

<div style="background:#E8EDF4;border-left:5px solid #0B1F3F;padding:16px 20px;border-radius:6px;margin:12px 0;"><strong style="color:#0B1F3F;">🎯 Learning Objective</strong><br><span style="color:#333;">Synthesise all analyses into a comprehensive model comparison table and outline the Decision Blueprint structure for the Session 18 capstone.</span></div>

<div style="background:#FDF6E8;border:2px solid #B8953E;padding:10px 18px;border-radius:8px;margin:20px 0 8px 0;"><h3 style="color:#0B1F3F;margin:0;">✏️ Exercise 13: Model Comparison Dashboard</h3></div>

In [ ]:
# YOUR CODE: Build a model comparison table with AUC-ROC, AUC-PR, Brier Score
# Rank models and visualise as a horizontal bar chart
# Which model would you recommend, and why?


<div style="background:linear-gradient(90deg,#E8EDF4,#F0E6F6);border:2px dashed #6C5B7B;padding:16px 20px;border-radius:8px;margin:16px 0;"><strong style="color:#6C5B7B;">🧭 What Would You Do Next?</strong><p style="color:#333;margin-top:8px;line-height:1.6;">You have 5 models, SHAP explanations, PDPs, calibration analysis, and a draft policy. If you were presenting to a Chief Risk Officer, how would you structure a 10-minute pitch? What would you show first? What question would you anticipate from a sceptical executive? This is exactly what the Session 18 capstone asks you to do.</p></div>

In [ ]:
# ══════════════════════════════════════════════════
#  SESSIONS 15\u201317 \u2014 FINAL SUMMARY
# ══════════════════════════════════════════════════

print("\u2550" * 70)
print("  SESSIONS 15\u201317: FROM MODELS TO DECISIONS \u2014 SUMMARY")
print("\u2550" * 70)

findings = [
    ("Best discriminator",           "LightGBM (highest AUC-ROC)"),
    ("Best calibrated",              "Logistic Regression (naturally calibrated)"),
    ("Best interpretable",           "Spline Logistic (near-GBM accuracy, fully transparent)"),
    ("Recommended for production",   "Spline Logistic or calibrated GBM (context-dependent)"),
    ("Top risk driver",              "EXT_SOURCE scores < 0.40"),
    ("Highest-risk segment",         "Tier E: ~5% of portfolio, ~35% of defaults"),
    ("Policy lever",                 "Rejecting Tier E saves ~35% of defaults"),
    ("Key deliverable",              "Decision Blueprint for Session 18 capstone"),
]

print(f"\n\u250C{'\u2500' * 68}\u2510")
for label, value in findings:
    print(f"\u2502  {label:<30s} {value:>35s} \u2502")
print(f"\u2514{'\u2500' * 68}\u2518")

print("\n\u2705 Sessions 15\u201317 Complete!")
print("\u27A1\uFE0F  Final Step: Session 18 Capstone \u2014 Build, Validate, and Present Your Decision Blueprint")

<div style="background:linear-gradient(135deg,#0B1F3F 0%,#008C8C 100%);padding:30px;border-radius:12px;margin-top:30px;">
<h2 style="color:white;font-family:Georgia,serif;margin:0;">
✅ Sessions 15–17 Complete
</h2>
<p style="color:#B8953E;font-size:1.15em;margin-top:10px;">
You can now build, compare, calibrate, and interpret credit-risk models. You can translate
model outputs into business policies with specific thresholds and adverse action templates.
You have everything needed for the capstone.
</p>
<p style="color:#B0D0D0;font-size:1em;margin-top:8px;">
<strong>Session 18 Capstone:</strong> Build your Decision Blueprint. It must include:<br>
• Data lineage and feature rationale (Sessions 13–14)<br>
• Model selection with performance comparison (Session 15)<br>
• Complexity trade-off analysis (Session 16)<br>
• SHAP-based policy recommendations (Session 17)<br>
• Risk tier framework and adverse action templates<br>
• Limitations, risks, and monitoring recommendations<br>
Present this as a 10-minute boardroom pitch. Peers will play sceptical executives.
</p>
</div>